# Middleware

docs: https://reference.langchain.com/python/langchain/middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = "GOOGLE_API_KEY"
GROQ_API_KEY = "GROQ_API_KEY"
OPENAI_API_KEY = "OPENAI_API_KEY"

def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None:
        raise ValueError(f"{name} is not set. Add it to your .env file.")
    os.environ[name] = value
    return value

google_api_key = require_env(GOOGLE_API_KEY)
groq_api_key = require_env(GROQ_API_KEY)
openai_api_key = require_env(OPENAI_API_KEY)

## Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b", # use cheaper model for summarization
            trigger=("messages", 10), # trigger when the messages length is 10
            keep=("messages", 4), #
        ),
    ]
)

In [6]:
### Run thread id
from langchain_core.runnables import RunnableConfig
config: RunnableConfig ={"configurable":{"thread_id": "test-1"}}

In [5]:
# test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d639ccf0-91ec-4a5d-8791-82d30198a1af'), AIMessage(content='$2 + 2 = 4$', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 19, 'total_tokens': 29, 'completion_time': 0.021478345, 'completion_tokens_details': None, 'prompt_time': 0.001080642, 'prompt_tokens_details': None, 'queue_time': 0.016693008, 'total_time': 0.022558987}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ee53-99e6-79d1-aca8-4d31bcd017d5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 10, 'total_tokens': 29})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d639ccf0-91ec-4a5d-8791-82d30198a1af'), AIMessage(

## Token Size



In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
# from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.runnables import RunnableConfig

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config: RunnableConfig = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [8]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~148 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='42716eb5-6440-4f39-909b-c26508a472b2'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'mvty8cknt', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 279, 'total_tokens': 306, 'completion_time': 0.072188308, 'completion_tokens_details': None, 'prompt_time': 0.019697532, 'prompt_tokens_details': None, 'queue_time': 0.022322333, 'total_time': 0.09188584}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ee58-386a-7df1-9687-74a193187344-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'mvty8cknt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

## Fraction


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# Φτιάχνεις το model object με profile
model = init_chat_model(
    "groq:qwen/qwen3.8-27b",
    profile={"max_input_tokens": 131072}  # βάλε το πραγματικό context window αυτού του μοντέλου
)

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~67 tokens (0.0523%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='6355f7a3-3379-424a-bdad-76972e59e07d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tfet0f6c6', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 271, 'total_tokens': 298, 'completion_time': 0.072871117, 'completion_tokens_details': None, 'prompt_time': 0.022961849, 'prompt_tokens_details': None, 'queue_time': 0.017562695, 'total_time': 0.095832966}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ee60-9c4a-7da3-8cb6-d275eba068dd-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'tfet0f6c6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'in


## Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import InterruptOnConfig
from langchain_core.messages import HumanMessage

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

interrupt_on: dict[str, bool | InterruptOnConfig] = {
    "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
    "read_email_tool": False,
}

agent=create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on=interrupt_on
        )
    ]
)

In [14]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [15]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f94bc66e-8ca8-460b-bc15-7e1f951c0302'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rpdpew0fw', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.18635663, 'completion_tokens_details': None, 'prompt_time': 0.032263823, 'prompt_tokens_details': None, 'queue_time': 0.022851558, 'total_time': 0.218620453}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f1d8-71ba-7080-8103-26a89a1a1f47-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'H

### Approve

In [16]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Email sent successfully to john@test.com with subject "Hello" and body "How are you?"


In [17]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f94bc66e-8ca8-460b-bc15-7e1f951c0302'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rpdpew0fw', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.18635663, 'completion_tokens_details': None, 'prompt_time': 0.032263823, 'prompt_tokens_details': None, 'queue_time': 0.022851558, 'total_time': 0.218620453}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f1d8-71ba-7080-8103-26a89a1a1f47-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'H

### Reject

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"



interrupt_on: dict[str, bool | InterruptOnConfig] = {
    "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
    "read_email_tool": False,
}


agent = create_agent(
       model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on=interrupt_on
        ),
    ],
)

In [20]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [21]:
from langgraph.types import Command
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It looks like the email send was rejected, so the message to john@test.com was not sent. Let me know if you'd like to make any changes before trying again.


### Editing

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"



interrupt_on: dict[str, bool | InterruptOnConfig] = {
    "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
    "read_email_tool": False,
}


agent = create_agent(
       model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on=interrupt_on
        ),
    ],
)

In [23]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='b357d6c4-f70f-4d7c-a601-916df1a72b92'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1wh40gcxc', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 374, 'total_tokens': 428, 'completion_time': 0.160477223, 'completion_tokens_details': None, 'prompt_time': 0.033490681, 'prompt_tokens_details': None, 'queue_time': 0.017714494, 'total_time': 0.193967904}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f1e3-fc24-7cc0-b823-c2886a3601d2-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'rec

In [25]:
from langgraph.types import Command
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email has been sent. Note that the recipient, subject, and body were edited by a human reviewer before sending — the email went to **correct@email.com** with subject **"Corrected Subject"** and body **"This was edited by human before sending"**.
